# ML4G Project 1: Getting Started

This notebook shows you how to **load the competition data** on Kaggle and how to **write a correctly formatted submission file**. The submission it produces is a constant placeholder, which scores 0. Your solution goes in the marked section.

**Before you start:** you may need to add the competition dataset to this notebook (right panel → *Add Input*), then run all cells.

## 1. Setup

In [1]:
import os
import glob
import zipfile

import numpy as np
import pandas as pd

## 2. Locate the data

Kaggle mounts attached datasets read-only under `/kaggle/input/`. If any files are still inside `.zip` archives, the cell below extracts them to `/kaggle/working/data`.

In [2]:
INPUT_ROOT = "/kaggle/input"
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
EXTRACT_DIR = os.path.join(WORK_DIR, "data")

for zp in glob.glob(os.path.join(INPUT_ROOT, "**", "*.zip"), recursive=True):
    target = os.path.join(EXTRACT_DIR, os.path.splitext(os.path.basename(zp))[0])
    if not os.path.isdir(target):
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(target)

ALL_FILES = sorted(
    p for root in [INPUT_ROOT, EXTRACT_DIR]
    for p in glob.glob(os.path.join(root, "**", "*"), recursive=True)
    if os.path.isfile(p) and not p.endswith(".zip")
)
print(f"{len(ALL_FILES)} files found")
for p in ALL_FILES:
    print(p)

52 files found
./data/CAGE-train/CAGE-train/X1_train_info.tsv
./data/CAGE-train/CAGE-train/X1_train_y.tsv
./data/CAGE-train/CAGE-train/X1_val_info.tsv
./data/CAGE-train/CAGE-train/X1_val_y.tsv
./data/CAGE-train/CAGE-train/X2_train_info.tsv
./data/CAGE-train/CAGE-train/X2_train_y.tsv
./data/CAGE-train/CAGE-train/X2_val_info.tsv
./data/CAGE-train/CAGE-train/X2_val_y.tsv
./data/CAGE-train/CAGE-train/X3_test_info.tsv
./data/DNase-bed/X1.bed
./data/DNase-bed/X2.bed
./data/DNase-bed/X3.bed
./data/DNase-bigwig/X1.bw
./data/DNase-bigwig/X2.bw
./data/DNase-bigwig/X3.bigwig
./data/H3K27ac-bed/X1.bed
./data/H3K27ac-bed/X2.bed
./data/H3K27ac-bed/X3.bed
./data/H3K27ac-bigwig/X1.bigwig
./data/H3K27ac-bigwig/X2.bw
./data/H3K27ac-bigwig/X3.bw
./data/H3K27me3-bed/X1.bed
./data/H3K27me3-bed/X2.bed
./data/H3K27me3-bed/X3.bed
./data/H3K27me3-bigwig/X1.bw
./data/H3K27me3-bigwig/X2.bw
./data/H3K27me3-bigwig/X3.bigwig
./data/H3K36me3-bed/X1.bed
./data/H3K36me3-bed/X2.bed
./data/H3K36me3-bed/X3.bed
./data/H3K

In [3]:
def find_file(name):
    # Path of the unique file whose name is `name`.
    hits = [p for p in ALL_FILES if os.path.basename(p) == name]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected exactly 1 file named {name}, found {len(hits)}: {hits}")
    return hits[0]


def find_track(mark, cell_line, kind):
    # Path of a chromatin track. kind: 'bed' (narrowPeak peaks) or 'bigwig' (continuous signal).
    exts = (".bed", ".bed.gz", ".narrowpeak", ".narrowpeak.gz") if kind == "bed" else (".bw", ".bigwig")
    hits = [
        p for p in ALL_FILES
        if mark in p and cell_line in os.path.basename(p) and p.lower().endswith(exts)
    ]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected exactly 1 {kind} file for {mark}/{cell_line}, found {len(hits)}: {hits}")
    return hits[0]

## 3. Gene information and targets

Each cell line and split has an `*_info.tsv` file with gene coordinates and, for X1 and X2, a `*_y.tsv` file with CAGE expression. The X3 test genes come without targets.

| Split | Chromosomes | Cell lines |
|---|---|---|
| train (set A) | chr2–13, chr15–18, chr20–22 | X1, X2 |
| val (set B) | chr14, chr19 | X1, X2 |
| test (set C) | chr1 | X3 |

In [4]:
def load_split(cell_line, split):
    info = pd.read_csv(find_file(f"{cell_line}_{split}_info.tsv"), sep="\t")
    if split == "test":
        return info
    y = pd.read_csv(find_file(f"{cell_line}_{split}_y.tsv"), sep="\t")
    return info.merge(y, on="gene_name", how="inner", validate="one_to_one")

data = {
    ("X1", "train"): load_split("X1", "train"),
    ("X1", "val"):   load_split("X1", "val"),
    ("X2", "train"): load_split("X2", "train"),
    ("X2", "val"):   load_split("X2", "val"),
    ("X3", "test"):  load_split("X3", "test"),
}

for key, df in data.items():
    print(key, df.shape)
data[("X1", "train")].head()

('X1', 'train') (14310, 8)
('X1', 'val') (1974, 8)
('X2', 'train') (14310, 8)
('X2', 'val') (1974, 8)
('X3', 'test') (1984, 7)


,gene_name,chr,gene_start,gene_end,TSS_start,TSS_end,strand,gex
0,SLC20A1,chr2,112645939,112663825,112658362,112658412,+,0.000000
1,C11orf58,chr11,16613132,16758340,16738643,16738693,+,2239.103328
2,ZSCAN9,chr6,28224886,28233487,28225263,28225313,+,19.798064
3,CD19,chr16,28931965,28939342,28931956,28932006,+,411.530623
4,TMEM123,chr11,102396332,102470384,102452789,102452839,-,34.214129


## 4. Chromatin tracks

Each assay (DNase, H3K4me3, H3K4me1, H3K27ac, H3K36me3, H3K27me3, H3K9me3) comes as **BED narrowPeak** (called peaks) and **bigWig** (continuous signal), one file per cell line.

narrowPeak files have 10 tab-separated columns without a header. Format specification: https://genome.ucsc.edu/FAQ/FAQformat.html#format12

In [5]:
NARROWPEAK_COLS = ["chrom", "chromStart", "chromEnd", "name", "score", "strand",
                   "signalValue", "pValue", "qValue", "peak"]

def load_peaks(mark, cell_line):
    return pd.read_csv(find_track(mark, cell_line, "bed"), sep="\t", header=None,
                       names=NARROWPEAK_COLS, usecols=range(10), comment="#")

peaks = load_peaks("DNase", "X1")
print(peaks.shape)
peaks.head()

(56452, 10)


,chrom,chromStart,chromEnd,name,score,strand,signalValue,pValue,qValue,peak
0,chr1,181405,181555,.,0,.,12.379543,-1,-1,75
1,chr1,267990,268140,.,0,.,12.379543,-1,-1,75
2,chr1,629160,629310,.,0,.,22.263896,-1,-1,75
3,chr1,629494,629644,.,0,.,10.732151,-1,-1,75
4,chr1,629870,630020,.,0,.,3603.694491,-1,-1,75


Reading bigWig files requires `pyBigWig`, which needs *Internet on* in the notebook settings (phone-verified Kaggle account):

In [6]:
# !pip install -q pyBigWig
# import pyBigWig
#
# bw = pyBigWig.open(find_track("DNase", "X1", "bigwig"))
# print(bw.chroms("chr2"))
# print(bw.stats("chr2", 1_000_000, 1_002_000, type="mean"))   # mean signal in a 2 kb window

## 5. Your solution

Build your features and model here. Train and validate on X1 and X2, then predict the expression of the X3 test genes.

The submission is scored with **Spearman's rank correlation**, so only the order of your predictions matters. You can compute it locally with `scipy.stats.spearmanr` on your validation sets.

At the end of this section, `pred` must be a numeric array with one value per row of `test_genes`, in the same order.

In [7]:
test_genes = data[("X3", "test")]

# ============================== YOUR SOLUTION ==============================
#
#
#
#
# ===========================================================================

# Placeholder: constant prediction (scores 0). Replace with your model's output.
pred = np.zeros(len(test_genes))

## 6. Write the submission

The required format is a CSV with header `gene_name,gex_predicted` and exactly one row per gene in `X3_test_info.tsv`, **without an index column**. The checks below catch the most common mistakes before you spend one of your 10 submissions.

In [8]:
submission = pd.DataFrame({
    "gene_name": test_genes["gene_name"].to_numpy(),
    "gex_predicted": np.asarray(pred, dtype=float),
})

assert len(submission) == len(test_genes), "One prediction per test gene required"
assert submission["gene_name"].is_unique, "Duplicate gene names"
assert np.isfinite(submission["gex_predicted"]).all(), "Predictions contain NaN or inf"

out_path = os.path.join(WORK_DIR, "submission.csv")
submission.to_csv(out_path, index=False)   # index=False is essential
print(f"Saved {out_path}")
print(open(out_path).read()[:150])

Saved ./submission.csv
gene_name,gex_predicted
CAPN9,0.0
ILF2,0.0
ST6GALNAC5,0.0
MROH7-TTC4,0.0
AGO4,0.0
AKR1A1,0.0
LRRC8B,0.0
EFNA3,0.0
GSTM3,0.0
CRYBG2,0.0
CCDC24,0.0
ARHG


### How to submit

- **From this notebook:** click *Save Version* (Save & Run All). When it finishes, open the version's *Output* tab and click *Submit*. The competition must be attached as an input for the button to appear.
- **Manually:** download `submission.csv` from the *Output* panel and upload it on the competition's *Submit Prediction* page.

Every successful submission counts toward your team's limit of 10. Files rejected with a format error don't count.